# ================================================================
#  MRI Demo-Fusion Ensemble — Single-Image Inference (Soft Vote)
# ================================================================

In [25]:
import sys
!{sys.executable} -m pip install torch --quiet


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:

import pickle, numpy as np, torch, torch.nn as nn, pandas as pd
from pathlib import Path

## test image

In [27]:
#IMAGE_ID   = "464357"                # HEALTHY EXAMPLE
#TRUE_LABEL = 0                       # 0=Healthy

IMAGE_ID   = "348321"                # ATROPHIC EXAMPLE
TRUE_LABEL = 1                       # 1=Atrophic

## architecture (from training notebook)

In [28]:
class Small3DCNNDemo(nn.Module):
    """Small3DCNN backbone + demographic MLP, fused before the head."""
    def __init__(self, n_demo_features=3, demo_hidden=16,
                 n_targets=1, dropout=0.3, channels=(16, 32, 64, 128)):
        super().__init__()
        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv3d(in_c, out_c, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm3d(out_c), nn.ReLU(inplace=True), nn.MaxPool3d(2)
            )
        layers = [block(1, channels[0])]
        for i in range(len(channels) - 1):
            layers.append(block(channels[i], channels[i + 1]))
        self.backbone     = nn.Sequential(*layers)
        self.global_pool  = nn.AdaptiveAvgPool3d(1)
        self.demo_encoder = nn.Sequential(
            nn.Linear(n_demo_features, demo_hidden), nn.ReLU(inplace=True),
            nn.Linear(demo_hidden, demo_hidden),     nn.ReLU(inplace=True),
        )
        self.dropout = nn.Dropout(dropout)
        self.head    = nn.Linear(channels[-1] + demo_hidden, n_targets)

    def forward(self, x, demo):
        feat = self.global_pool(self.backbone(x)).flatten(1)
        d    = self.demo_encoder(demo)
        z    = self.dropout(torch.cat([feat, d], dim=1))
        return self.head(z).squeeze(1)

## load models

In [29]:
# ── CONFIG ──────────────────────────────────────────────────────
FOLD_DIR   = Path("models")
IMAGE_DIR  = Path("images")
CSV_PATH   = Path("data/data.csv")   # columns: image_id, AGE, PTGENDER, PTEDUCAT
# ────────────────────────────────────────────────────────────────

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_FOLDS = 7
CHANNEL_CONFIGS = {
    "narrow_3":   (8,  16,  32),
    "narrow_4":   (8,  16,  32,  64),
    "standard_4": (16, 32,  64, 128),
    "wide_4":     (32, 64, 128, 256),
}


# ── Load subject info from CSV ───────────────────────────────────
df = pd.read_csv(CSV_PATH)
df["image_id"] = df["image_id"].astype(str).str.replace(".npy", "", regex=False)
row = df[df["image_id"] == IMAGE_ID.replace(".npy", "")].iloc[0]

AGE, SEX, EDU = float(row["AGE"]), str(row["PTGENDER"]), float(row["PTEDUCAT"])
print(f"Subject: {IMAGE_ID}  AGE={AGE}  SEX={SEX}  EDU={EDU}")

# ── Load image ───────────────────────────────────────────────────
img = np.load(IMAGE_DIR / f"{IMAGE_ID}.npy").astype(np.float32)
x   = torch.from_numpy(img[None, None]).to(DEVICE)

Subject: 348321  AGE=76.7  SEX=0  EDU=16.0


## Ensemble inference

In [30]:
fold_probs, fold_thresholds = [], []

for k in range(N_FOLDS):
    with open(FOLD_DIR / f"fold_{k}.pkl", "rb") as f:
        res = pickle.load(f)
    scaler = res["demo_scaler"]
    cfg    = res["config"]
    thr    = res["test_results"]["threshold"]

    gender = 1.0 if SEX.strip().upper().startswith("F") else 0.0
    age_z  = (AGE - scaler["age_mean"]) / scaler["age_std"]
    edu_z  = (EDU - scaler["edu_mean"]) / scaler["edu_std"]
    demo   = torch.tensor([[gender, age_z, edu_z]], dtype=torch.float32).to(DEVICE)

    model = Small3DCNNDemo(
        n_demo_features=3, demo_hidden=cfg["demo_hidden"],
        n_targets=1, dropout=cfg["dropout"],
        channels=CHANNEL_CONFIGS[cfg["channels_key"]]
    ).to(DEVICE)
    model.load_state_dict(torch.load(FOLD_DIR / f"fold_{k}_best_model.pt", map_location=DEVICE))
    model.eval()
    with torch.no_grad():
        prob = torch.sigmoid(model(x, demo)).item()

    fold_probs.append(prob)
    fold_thresholds.append(thr)
    print(f"  Fold {k}: prob={prob:.3f}  thr={thr:.3f}  → {'ATROPHIC' if prob >= thr else 'healthy'}")
    del model; torch.cuda.empty_cache()

  Fold 0: prob=0.416  thr=0.355  → ATROPHIC
  Fold 1: prob=0.818  thr=0.645  → ATROPHIC
  Fold 2: prob=0.861  thr=0.595  → ATROPHIC
  Fold 3: prob=0.874  thr=0.535  → ATROPHIC
  Fold 4: prob=0.948  thr=0.795  → ATROPHIC
  Fold 5: prob=0.669  thr=0.715  → healthy
  Fold 6: prob=0.370  thr=0.300  → ATROPHIC


## Soft vote 

In [31]:
avg_prob   = float(np.mean(fold_probs))
mean_thr   = float(np.mean(fold_thresholds))
prediction = int(avg_prob >= mean_thr)

print("\n" + "="*50)
print(f"  Ensemble avg prob : {avg_prob:.3f}")
print(f"  Mean threshold    : {mean_thr:.3f}")
print(f"  Prediction        : {'ATROPHIC (1)' if prediction else 'HEALTHY (0)'}")
if TRUE_LABEL is not None:
    print(f"  True label        : {'ATROPHIC (1)' if TRUE_LABEL else 'HEALTHY (0)'}")
    print(f"  Result            : {'✓ CORRECT' if prediction == TRUE_LABEL else '✗ WRONG'}")
print("="*50)


  Ensemble avg prob : 0.708
  Mean threshold    : 0.563
  Prediction        : ATROPHIC (1)
  True label        : ATROPHIC (1)
  Result            : ✓ CORRECT
